# Lab: Coding Cross-Validation and Forward Selection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okuchap/GB656_2026_public/blob/main/problem-sets/lab-lectures/05-04-lab.ipynb)

This lab prepares you for Problem Set 4. It focuses on the scientific-computing workflow rather than re-teaching the model-selection theory from lecture. We will predict vehicle fuel efficiency with Auto MPG, a different outcome and dataset from the graded insurance task.

By the end, you should be able to:

- visualize a numeric outcome with a histogram;
- create baseline and engineered feature tables;
- fit a scikit-learn linear regression and calculate training RMSE;
- define reproducible five-fold splits with `KFold`;
- convert scikit-learn's negative MSE scores into fold RMSE values;
- average fold RMSE values and compare candidate models;
- trace and run a forward-selection algorithm; and
- distinguish a model-selection score from an untouched final assessment.

## How to use this lab

Open the lab using the course Google Colab link and work from top to bottom. Read the explanation before each code cell, predict what the cell will return, and then run it. Complete the short exercises as you go. At the end, select **Runtime > Restart session and run all** to make sure the workflow does not depend on hidden notebook state.

You do not submit this lab. If you want your changes to persist after you close Colab, select **File > Save a copy in Drive**; otherwise, saving a copy is optional.

## 0. Import packages

Google Colab already includes these packages in a standard runtime. The model-selection tools come from scikit-learn. NumPy handles square roots and repeated values, pandas manages tables, and Matplotlib makes the exploratory plot. `Path` and `quote` support the local-or-public data-loading helper.

In [ ]:
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score

plt.style.use("seaborn-v0_8-whitegrid")

The three scikit-learn imports have separate jobs:

- `LinearRegression` creates an ordinary least-squares prediction model with an intercept by default.
- `mean_squared_error` compares observed and predicted outcomes.
- `KFold` defines reusable data splits, while `cross_val_score` refits and evaluates a fresh model in every fold.

## 1. Define two small helper functions

RMSE is the square root of mean squared error. It is expressed in the same units as the outcome, which will be miles per gallon in this lab.

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

The instructor-provided helper below first searches the current directory and its parent directories for the course data file. If it cannot find a local copy—as in a fresh Colab runtime—it returns the raw-data URL from the public course repository on GitHub. You do not need to upload the CSV or mount Google Drive.

In [ ]:
PUBLIC_REPOSITORY = "okuchap/GB656_2026_public"
PUBLIC_REVISION = "main"


def course_data_source(file_name):
    """Return a local course-data path when available, otherwise its public URL."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local_path = root / "data" / file_name
        if local_path.is_file():
            return local_path

    encoded_name = quote(file_name)
    return (
        "https://raw.githubusercontent.com/"
        f"{PUBLIC_REPOSITORY}/{PUBLIC_REVISION}/data/{encoded_name}"
    )

## 2. Load and clean Auto MPG

Each row is a vehicle. The outcome `mpg` measures fuel efficiency. Load the local or public course copy and check its dimensions before changing it.

In [ ]:
auto_source = course_data_source("auto-mpg.csv")
auto_raw = pd.read_csv(auto_source)
source_location = (
    "local course repository" if isinstance(auto_source, Path) else "public GitHub repository"
)
print(
    f"Loaded {auto_raw.shape[0]:,} rows and {auto_raw.shape[1]} columns "
    f"from the {source_location}."
)

auto_raw.shape

You should see 398 rows and 9 columns.

### Check for data-quality issues

Before cleaning the data, inspect the columns for unusual values or missing-value codes. In this dataset, `horsepower` is stored as text rather than numeric values, so first examine its distinct values and identify entries that cannot be interpreted as numbers.

In [ ]:
auto_raw.head()

In [ ]:
auto_raw.info()

Pay attention to the unusual values in `horsepower`.

In [ ]:
auto_raw.loc[pd.to_numeric(auto_raw["horsepower"], errors="coerce").isna(), "horsepower"]

The inspection reveals six rows where `horsepower` is stored as the string `?`. These are missing values encoded as text, which is why pandas does not initially treat the column as numeric.

We will remove those six rows, convert the remaining horsepower values to numeric form, rename `model year` to `model_year`, and keep the six columns needed for the analysis. Each operation is assigned back to the working DataFrame so later cells use the cleaned version.

In [ ]:
auto = auto_raw.copy()
auto = auto.loc[auto["horsepower"] != "?"].copy()
auto["horsepower"] = pd.to_numeric(auto["horsepower"])
auto = auto.rename(columns={"model year": "model_year"})

analysis_columns = [
    "mpg",
    "horsepower",
    "weight",
    "displacement",
    "acceleration",
    "model_year",
]
auto = auto[analysis_columns].dropna().reset_index(drop=True)

auto.shape

### Exercise 1 — verify the analysis table

Before modeling, check that cleaning produced 392 complete rows, six intended columns, and a numeric horsepower column. Assertions stop the notebook immediately if an assumption is false.

In [ ]:
assert auto.shape == (392, 6)
assert auto.columns.tolist() == analysis_columns
assert pd.api.types.is_numeric_dtype(auto["horsepower"])
assert auto.notna().all().all()

print("Analysis table checks passed.")

## 3. Inspect a pattern before engineering features

Summary statistics check scales and ranges. They also make unit mistakes easier to notice later.

In [ ]:
auto[analysis_columns].describe().round(2)

### Visualize one numeric variable with a histogram

A histogram summarizes the shape of one numeric variable. `ax.hist(auto["mpg"], bins=20, ...)` divides the observed MPG range into 20 intervals, called bins, and draws the number of vehicles in each interval. Too few bins can hide important shape, while too many can emphasize small sample fluctuations, so the bin count is a display choice rather than a model setting.

The horizontal axis should name the variable and its unit; the vertical axis reports the number of observations. The `edgecolor` argument makes adjacent bars easier to distinguish. Unlike a scatterplot, a histogram describes one variable and does not show its relationship with a predictor.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(
    auto["mpg"],
    bins=20,
    color="#4C78A8",
    edgecolor="white",
)
ax.set_xlabel("Miles per gallon")
ax.set_ylabel("Number of vehicles")
ax.set_title("Distribution of vehicle fuel efficiency")

plt.show()

A scatterplot can suggest transformations. Put horsepower on the horizontal axis and MPG on the vertical axis because horsepower will be a predictor and MPG the outcome.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(auto["horsepower"], auto["mpg"], alpha=0.45)
ax.set_xlabel("Horsepower")
ax.set_ylabel("Miles per gallon")
ax.set_title("Fuel efficiency and horsepower")

plt.show()

### Exercise 2 — read the plot

The relationship is negative and curved: MPG falls quickly across lower horsepower values and then flattens. A squared horsepower term is therefore a plausible candidate feature.

However, seeing curvature in this sample does not guarantee that adding the squared term will improve predictions for new vehicles. The extra flexibility may capture a real relationship, or it may partly fit patterns that are specific to this sample. Later, we will use cross-validation to evaluate whether the added feature improves out-of-sample prediction.

## 4. Create baseline and engineered feature sets

A feature list is just a list of column names. Passing a list into `X[features]` lets one feature table support many candidate models without copying the data. Before squaring or multiplying large measurements, put them in convenient units. This keeps the design matrix numerically well-scaled without changing the predictive information.

In [ ]:
auto["horsepower_100"] = auto["horsepower"] / 100
auto["weight_1000"] = auto["weight"] / 1_000
auto["displacement_100"] = auto["displacement"] / 100

baseline_features = [
    "horsepower_100",
    "weight_1000",
    "displacement_100",
    "acceleration",
    "model_year",
]

Squared terms let a linear regression represent curvature. The product `horsepower_100_weight_1000` is an interaction: it lets the relationship involving one predictor depend on the other predictor's value. These are numeric columns even though the fitted model remains linear in its coefficients.

In [ ]:
auto["horsepower_100_squared"] = auto["horsepower_100"] ** 2
auto["weight_1000_squared"] = auto["weight_1000"] ** 2
auto["horsepower_100_weight_1000"] = (
    auto["horsepower_100"] * auto["weight_1000"]
)

engineered_features = [
    "horsepower_100_squared",
    "weight_1000_squared",
    "horsepower_100_weight_1000",
]

Combine the name lists, then create one feature table `X` and one outcome Series `y`. Scikit-learn will add the regression intercept automatically.

In [ ]:
all_features = baseline_features + engineered_features
X = auto[all_features]
y = auto["mpg"]

X.head()

### Exercise 3 — check model inputs

A model needs one outcome per feature-table row, unique feature names, numeric values, and no missing values. Run these checks before a long model search.

In [ ]:
assert len(X) == len(y) == 392
assert X.columns.is_unique
assert all(pd.api.types.is_numeric_dtype(X[column]) for column in X.columns)
assert X.notna().all().all()

print("Model-input checks passed.")

## 5. Calculate training RMSE

`.fit()` estimates coefficients from the supplied rows. `.predict()` then applies the fitted rule. Here both methods use the same baseline feature columns and all 392 rows.

In [ ]:
baseline_model = LinearRegression()
baseline_model.fit(X[baseline_features], y)

training_predictions = baseline_model.predict(X[baseline_features])
baseline_training_rmse = rmse(y, training_predictions)
baseline_training_rmse

The result should be a little above 3 MPG. This is an in-sample fit measure, not an estimate from held-out vehicles. Because the coefficients were chosen to fit these same rows, training RMSE is usually optimistic.

## 6. Define and inspect five folds

`KFold` records how to divide row positions. `n_splits=5` creates five validation folds; `shuffle=True` mixes row order first; and `random_state=505` makes that shuffle reproducible. Reusing this same object gives every candidate model the same comparison rows.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=505)

`.split(X)` yields training and validation row-position arrays for one fold at a time. The loop below records only their sizes so you can see the rotation without printing hundreds of positions.

In [ ]:
fold_rows = []

for fold_number, (train_index, validation_index) in enumerate(kf.split(X), start=1):
    fold_rows.append(
        {
            "fold": fold_number,
            "training_rows": len(train_index),
            "validation_rows": len(validation_index),
        }
    )

fold_table = pd.DataFrame(fold_rows)
fold_table

### Check the folds

There are five rows. Each validation fold has 78 or 79 vehicles, and each training fold has the remaining 313 or 314. Every vehicle is held out once and used for training four times.

## 7. Convert cross-validation scores to fold RMSE

Scikit-learn uses a convention that every score should be maximized. Because lower MSE is better, the scoring name is `neg_mean_squared_error`, and the returned values are negative. Those negative values are not negative prediction errors: multiply by -1 to recover MSE, then take the square root to recover RMSE.

In [ ]:
negative_mse = cross_val_score(
    LinearRegression(),
    X[baseline_features],
    y,
    cv=kf,
    scoring="neg_mean_squared_error",
)

fold_mse = -negative_mse
fold_rmse = np.sqrt(fold_mse)

fold_score_table = pd.DataFrame(
    {
        "fold": np.arange(1, 6),
        "negative_mse_from_sklearn": negative_mse,
        "mse": fold_mse,
        "rmse": fold_rmse,
    }
)
fold_score_table.round(3)

The course reports **mean fold RMSE**: take each fold's square root first, then average. This gives each fold one RMSE contribution. `np.sqrt(fold_mse.mean())` is a different aggregation and is not the requested measure.

In [ ]:
baseline_mean_cv_rmse = fold_rmse.mean()
baseline_mean_cv_rmse

### Exercise 4 — verify the conversion

The original scikit-learn values should be nonpositive, the five RMSE values should be positive, and their stored mean should match the value just displayed.

In [ ]:
assert len(negative_mse) == 5
assert np.all(negative_mse <= 0)
assert np.all(fold_rmse > 0)
assert np.isclose(baseline_mean_cv_rmse, fold_score_table["rmse"].mean())

print("Cross-validation score conversion checks passed.")

## 8. Package the repeated scoring workflow

Model comparison repeats the same operations many times. `cv_rmse()` accepts a feature-name collection, selects those columns, refits a new `LinearRegression` in every fold, and returns one RMSE per fold. Its empty-feature branch creates a mean-only starting model for forward selection.

In [ ]:
def cv_rmse(features, X, y, cv):
    features = list(features)

    if not features:
        fold_rmse = []
        for train_index, validation_index in cv.split(X):
            y_train = y.iloc[train_index]
            y_validation = y.iloc[validation_index]
            mean_predictions = np.repeat(y_train.mean(), len(validation_index))
            fold_rmse.append(rmse(y_validation, mean_predictions))
        return np.asarray(fold_rmse)

    negative_mse = cross_val_score(
        LinearRegression(),
        X[features],
        y,
        cv=cv,
        scoring="neg_mean_squared_error",
    )
    return np.sqrt(-negative_mse)

`score_model()` fits one model on all rows for training RMSE and separately calls `cv_rmse()` for held-out scores. The returned dictionary has stable names that can become columns in a comparison table.

In [ ]:
def score_model(features, X, y, cv):
    features = list(features)
    if not features:
        raise ValueError("score_model() requires at least one feature.")

    model = LinearRegression()
    model.fit(X[features], y)
    training_predictions = model.predict(X[features])
    fold_rmse = cv_rmse(features, X, y, cv)

    return {
        "training_rmse": rmse(y, training_predictions),
        "mean_cv_rmse": fold_rmse.mean(),
    }

Call the function on the baseline feature-name list. It should reproduce the separately calculated training and CV values.

In [ ]:
baseline_scores = score_model(baseline_features, X, y, kf)
baseline_scores

### Exercise 5 — check the reusable function

Verify that packaging the workflow did not change the answer. A failed assertion usually means the feature list, fold object, or order of the RMSE operations changed.

In [ ]:
assert np.isclose(baseline_scores["training_rmse"], baseline_training_rmse)
assert np.isclose(baseline_scores["mean_cv_rmse"], baseline_mean_cv_rmse)

print("Reusable scoring function matches the step-by-step calculation.")

## 9. Compare a small set of candidate models

A dictionary maps readable model labels to feature-name lists. The loop scores each list and appends one row. The resulting table keeps training and validation measures separate.

In [ ]:
candidate_feature_sets = {
    "Baseline": baseline_features,
    "Baseline + horsepower squared": baseline_features + ["horsepower_100_squared"],
    "All engineered features": all_features,
}

comparison_rows = []

for model_name, features in candidate_feature_sets.items():
    scores = score_model(features, X, y, kf)
    comparison_rows.append({"model": model_name, **scores})

candidate_table = pd.DataFrame(comparison_rows).set_index("model")
candidate_table.round(3)

### Exercise 6 — read the candidate table

Identify the lowest training RMSE and lowest mean CV-RMSE. They need not occur in the same row.

Adding predictors cannot increase ordinary least-squares training error because the model has more flexibility to fit the training data. However, the CV-RMSE can rise if the additional flexibility begins to fit sample-specific noise rather than patterns that generalize to new observations. This is an example of **overfitting**: a more complex model can fit the observed sample better while predicting new data worse.

## 10. Search a forward-selection path

Forward selection is a greedy search. In plain language:

1. Start with no features.
2. Try adding each remaining encoded column.
3. Keep the candidate addition with the lowest mean CV-RMSE.
4. Record that model as the next path row.
5. Repeat until every column has been added.

The best candidate at one step need not improve on the previous step, and the greedy path does not examine every possible subset.

The label helper joins selected column names for readable output. It labels the empty starting set as a mean-only model.

In [ ]:
def feature_set_label(features):
    return ", ".join(features) if features else "(mean only)"

Read the next function from the outside in. The outer loop creates one path step. The inner loop scores every possible one-column addition. `min(..., key=...)` chooses the candidate with the lowest mean CV-RMSE. Selected columns are stored as a tuple so they remain one object inside a DataFrame cell.

In [ ]:
def forward_selection_cv(X, y, feature_pool, cv):
    selected_features = []
    remaining_features = list(feature_pool)
    path_rows = [
        {
            "step": 0,
            "added_feature": "(start)",
            "features": tuple(),
            "feature_label": "(mean only)",
            "mean_cv_rmse": cv_rmse([], X, y, cv).mean(),
        }
    ]

    for step in range(1, len(feature_pool) + 1):
        candidate_rows = []

        for feature in remaining_features:
            candidate_features = selected_features + [feature]
            candidate_rows.append(
                {
                    "feature": feature,
                    "features": candidate_features,
                    "mean_cv_rmse": cv_rmse(
                        candidate_features, X, y, cv
                    ).mean(),
                }
            )

        best_candidate = min(
            candidate_rows, key=lambda row: row["mean_cv_rmse"]
        )
        selected_features = best_candidate["features"]
        remaining_features.remove(best_candidate["feature"])

        path_rows.append(
            {
                "step": step,
                "added_feature": best_candidate["feature"],
                "features": tuple(selected_features),
                "feature_label": feature_set_label(selected_features),
                "mean_cv_rmse": best_candidate["mean_cv_rmse"],
            }
        )

    return pd.DataFrame(path_rows)

Run the search on all eight encoded numeric columns. The returned table contains the mean-only start plus one row per added column. The temporary `pd.option_context(...)` setting prevents pandas from shortening the selected-feature labels with an ellipsis; it applies only while this display is created.

In [ ]:
forward_path = forward_selection_cv(X, y, all_features, kf)

forward_path_display = forward_path[
    ["step", "added_feature", "feature_label", "mean_cv_rmse"]
].round(3)

with pd.option_context("display.max_colwidth", None):
    display(forward_path_display)

`.idxmin()` returns the index label of the smallest value in a Series. Use that label with `.loc` to retrieve the whole best row. Do not assume the last, largest model is best.

In [ ]:
best_path_index = forward_path["mean_cv_rmse"].idxmin()
forward_selected = forward_path.loc[best_path_index]
forward_features = list(forward_selected["features"])
omitted_features = [
    feature for feature in all_features if feature not in forward_features
]

print("Selected step:", int(forward_selected["step"]))
print("Selected features:", forward_features)
print("Omitted features:", omitted_features)
print(f"Selected mean CV-RMSE: {forward_selected['mean_cv_rmse']:.3f} MPG")

### Exercise 7 — trace the selection

The next display calculates the change from each previous path row. A negative change means the new row has lower mean CV-RMSE; a positive change means it became worse. Find one improvement and, if present, one worsening.

In [ ]:
path_diagnostics = forward_path[["step", "added_feature", "mean_cv_rmse"]].copy()
path_diagnostics["change_from_previous"] = (
    path_diagnostics["mean_cv_rmse"].diff()
)

path_diagnostics.round(3)

The minimum path score was used to choose the feature set. Reporting that same minimum as if it came from untouched data would ignore selection optimism. For a final assessment, keep a test set completely outside the search or repeat the entire selection inside nested cross-validation.

## 11. Add the selected model to the comparison

Score the selected feature list on the same folds and add one final row. The training and CV columns answer different questions and should remain separate.

In [ ]:
forward_scores = score_model(forward_features, X, y, kf)

final_comparison = candidate_table.copy()
final_comparison.loc["Forward-selection model"] = forward_scores
final_comparison.round(3)

### Exercise 8 — explain the result

In two or three sentences, identify the model with the lowest training RMSE and the model with the lowest mean CV-RMSE. If they differ, explain why that is possible. Then state why the selected model still needs untouched assessment before making a high-stakes performance claim.

## 12. Common mistakes and quick diagnoses

| Symptom | Likely cause | Check |
|---|---|---|
| Data loading raises an HTTP or file error | Neither a local course copy nor the public GitHub copy is reachable | Reopen the course Colab link, confirm that the runtime has internet access, and rerun the setup and loading cells |
| Conversion error involving `?` | Raw `horsepower` strings reached the model | Run the cleaning cell and check the dtype |
| `Input X contains NaN` | Missing rows or engineered values were not checked | Run the model-input assertions |
| Square root warning or `nan` RMSE | Negative scikit-learn scores were not negated | Use `np.sqrt(-negative_mse)` |
| Different model comparisons on each run | Folds were shuffled without a fixed seed | Reuse `KFold(..., random_state=505)` |
| A candidate is missing a column | Feature name was misspelled or omitted from its list | Compare names with `X.columns.tolist()` |
| Selected model is always the largest model | Last path row was used instead of the minimum | Use `.idxmin()` and `.loc` |
| Notebook works only after running cells out of order | Hidden session state from an earlier run | Restart the session and run all cells |

## 13. Transfer the workflow to Problem Set 4

The graded task changes the business context but preserves the workflow:

| This lab | Problem Set 4 |
|---|---|
| `mpg` | `charges` |
| Auto MPG numeric columns | Encoded insurance predictor columns |
| horsepower/weight transformations | age/BMI transformations and BMI-smoking interaction |
| one `KFold` object | the same specified `KFold` object |
| candidate and forward-path tables | baseline, engineered, and forward comparison |

Earlier labs introduced dummy encoding. Problem Set 4 specifies the category baselines, and you will apply the earlier `pd.get_dummies(..., drop_first=True)` workflow yourself. Your new model-selection responsibilities are to keep the encoded feature definitions fixed across candidates, choose one additional predictor-derived numeric feature, and apply the CV/selection workflow from this lab.

### Final transfer exercise

Without copying code yet, write down answers to these questions:

1. Which object must stay identical across every model comparison?
2. Why must your additional insurance feature avoid `charges`?
3. Which operation finds the selected forward-path row?
4. Why is that row's CV-RMSE not an untouched final assessment?

Then restart this lab's session and run all cells. If every assertion passes, you are ready to adapt the workflow.